# La Grulla Blanca — Metal Retest v0.1
Sentencia + Mandato variants, plus regression for Forja, Espejo and Tormenta.

Authority: `NEW_COMBAT_STATS_V0_1`. LAB only; no canon/runtime/merge.

Output to return: `METAL_RETEST_RESULTS_FOR_CHATGPT.zip`.


In [ ]:
from pathlib import Path
import base64, hashlib, json, os, shutil, urllib.parse, urllib.request, zipfile

OWNER="Shein25"
REPO="La-Grulla-Blanca"
BRANCH="experiment/ultis-metal-retest-v0.1"
ROOT_PATH="experimentos/ultis/metal_retest_v01/package_chunks_v01"
EXPECTED_SHA256="01350244699edefa904e42713e1e5b24f901ba5dc985b829219ffcf60e45f178"

def github_text(path):
    ref=urllib.parse.quote(BRANCH,safe="")
    path_q="/".join(urllib.parse.quote(x,safe="") for x in path.split("/"))
    url=f"https://api.github.com/repos/{OWNER}/{REPO}/contents/{path_q}?ref={ref}"
    req=urllib.request.Request(url,headers={"Accept":"application/vnd.github+json","User-Agent":"La-Grulla-Blanca-Metal-Retest/1.0"})
    with urllib.request.urlopen(req,timeout=60) as r: obj=json.loads(r.read().decode("utf-8"))
    if obj.get("encoding")!="base64": raise RuntimeError(f"Unexpected GitHub encoding for {path}: {obj.get('encoding')}")
    return base64.b64decode(obj["content"]).decode("utf-8")

manifest=json.loads(github_text(f"{ROOT_PATH}/manifest.json"))
joined="".join(github_text(f"{ROOT_PATH}/{name}").strip() for name in manifest["parts"])
if len(joined)!=manifest["base64_length"]: raise RuntimeError(f"Base64 length mismatch: {len(joined)} != {manifest['base64_length']}")
data=base64.b64decode(joined,validate=True)
sha=hashlib.sha256(data).hexdigest()
if len(data)!=manifest["decoded_size_bytes"]: raise RuntimeError("ZIP size mismatch")
if sha!=EXPECTED_SHA256 or sha!=manifest["sha256"]: raise RuntimeError(f"SHA mismatch: {sha}")
ROOT=Path("/kaggle/working/metal_retest_pkg")
if ROOT.exists(): shutil.rmtree(ROOT)
ROOT.mkdir(parents=True,exist_ok=True)
zp=Path("/kaggle/working")/manifest["package"]
zp.write_bytes(data)
with zipfile.ZipFile(zp) as z:
    bad=z.testzip()
    if bad: raise RuntimeError(f"Corrupt ZIP member: {bad}")
    z.extractall(ROOT)
print("GIT_PACKAGE_OK")
print("revision:",manifest["revision"])
print("SHA-256:",sha)
print("scenarios:",manifest["scenario_count"])
print("runner:",ROOT/"run_metal_retest_kaggle.py")


In [ ]:
# Mandatory smoke before the heavy run.
SMOKE=Path("/kaggle/working/metal_retest_smoke")
cmd=f"python {ROOT/'run_metal_retest_kaggle.py'} --mode smoke --runs 500 --workers 0 --output {SMOKE}"
print(cmd)
rc=os.system(cmd)
if rc!=0: raise RuntimeError(f"Smoke failed: {rc}")
check=json.loads((SMOKE/"SELF_CHECK.json").read_text())
if not check.get("pass"): raise RuntimeError(f"SELF_CHECK failed: {check}")
print("SMOKE + SELF_CHECK: PASS")


In [ ]:
# Heavy: 69 scenarios x 100,000 = 6,900,000 simulations.
RUNS=100_000
OUT=Path("/kaggle/working/metal_retest_results")
cmd=f"python {ROOT/'run_metal_retest_kaggle.py'} --mode heavy --runs {RUNS} --workers 0 --output {OUT}"
print(cmd)
rc=os.system(cmd)
if rc!=0: raise RuntimeError(f"Heavy runner failed: {rc}")


In [ ]:
from IPython.display import FileLink, display
result=OUT/"METAL_RETEST_RESULTS_FOR_CHATGPT.zip"
print("Compact result:",result,f"{result.stat().st_size/1024:.1f} KiB")
display(FileLink(str(result)))
print("Return only that ZIP to ChatGPT.")
